In [ ]:
#Python
#import re
#import random
import importlib
from collections import Counter
import unicodedata
#Terceros
import pandas as pd
import numpy as np
#Este proyecto
import set_paths
import config.paths as pths 
from config.patrones import Pattern, PATRONES, abrev_dict
from src.replace_abrev import tokenizar_direccion, reemplazar_abrevs 
from src.elim_noise import limpiar_ruido, normalizar, match, redundant_patterns
from src.cleaning_pipe import clean
from src.pipeline import join_tokens


## TEST REEMPLAZO ABREVIATURAS

In [2]:
#Test
direcciones = pd.read_csv(pths.DIRECCIONES_CLEAN / "universo_direcciones_clean.csv")
direcciones_test = direcciones["DIRECCIÓN GARANTÍA"].sample(n=10, random_state=1105)
for direccion in direcciones_test:
    tokens = tokenizar_direccion(direccion)
    reemplazadas = reemplazar_abrevs(tokens, abrev_dict)
    print(f"{direccion}")
    print(reemplazadas)

CALLE FLAMBOYAN LOTE 17 MANZANA 001 SM 002
['Calle', 'FLAMBOYAN', 'Lt.', '17', 'Mz.', '001', 'SM', '002']
MANZANA 53 LOTE 198 CALLE RET  8 CIELO CASA D
['Mz.', '53', 'Lt.', '198', 'Calle', 'RET', '8', 'CIELO', 'CASA', 'D']
CALLE BETELGUESE NO 106 LT 23 MZ 05
['Calle', 'BETELGUESE', ' No.', '106', 'Lt.', '23', 'Mz.', '05']
Av. Hidalgo , Dpto. 101, Ed. 6 No. 1
['Av.', 'Hidalgo', 'Dpto.', '101', 'Ed.', '6', 'No.', '1']
CERRADA DEL RAYO  OFICIAL 29-A LOTE 29 LA VILLITA
['CERRADA', 'DEL', 'RAYO', 'OFICIAL', '29', '-', 'A', 'Lt.', '29', 'LA', 'VILLITA']
Torre Jacarandas, Condominio 178, Conjunto Condominal Viva Mx, Centlapal identif
['Torre', 'Jacarandas', 'Condominio', '178', 'Conjunto', 'Condominal', 'Viva', 'Mx', 'Centlapal', 'identif']
PRIVADA LAS FLORES, NUM EXT 2, M-6, L-6
['PRIVADA', 'LAS', 'FLORES', 'No.', 'Ext.', '2', 'Mz.', '-', '6', 'Lt.', '-', '6']
MZ 51 LTE 2 C. HDA LA JALAPILLA N NUM 15
['Mz.', '51', 'Lt.', '2', 'Calle', 'HDA', 'LA', 'JALAPILLA', ' No.', 'No.', '15']
BOULEVARD 

## TEST N-GRAMAS

In [3]:
direcciones["tokens"] = direcciones["DIRECCIÓN GARANTÍA"].astype(str).apply(lambda x:reemplazar_abrevs(tokenizar_direccion(x), abrev_dict))

In [4]:
counters = {
    1: Counter(),
    2: Counter(),
    3: Counter(),
    4: Counter(),
    5: Counter(),
    8: Counter()
}

for tokens in direcciones["tokens"]:
    for n, counter in counters.items():
        for i in range(len(tokens)-n+1):
            counter[tuple(tokens[i:i+n])] += 1 # -> Construir N-gramas 
#counters[5].most_common() -> Patrones más comunes por longitud (5)

## TEST ENCONTRAR FRASES

In [5]:
np.random.seed(1105)
test_adress = np.random.randint(0, direcciones.shape[0]-1)
print(direcciones["DIRECCIÓN GARANTÍA"][test_adress])
print(limpiar_ruido(tokens = direcciones["tokens"][test_adress]))
print("")

3RA CDA FELIPE ANGELES MZA A LT 9
['3', 'RA', 'CDA', 'FELIPE', 'ANGELES', 'Mz.', 'A', 'Lt.', '9']



## FULL

In [6]:
for i in range(30):
    np.random.seed(int(np.log(1105)+i))
    test_adress = np.random.randint(0, direcciones.shape[0]-1)
    print(f"{test_adress}: {direcciones["DIRECCIÓN GARANTÍA"][test_adress]}")
    print(clean(raw_adress = direcciones["DIRECCIÓN GARANTÍA"][test_adress]))
    #time.sleep(10)

10742: SAN MATEO 7844
['SAN', 'MATEO', '7844']
4547: PRIVADA TRECE No. 58 LOTE 10 MANZANA 14 
['PRIVADA', 'TRECE', 'No.', '58', 'Lt.', '10', 'Mz.', '14']
20828: VALLE FERTIL 16 16 D
['VALLE', 'FERTIL', '16', '16', 'D']
17673: C PALMA MZ 34 LT 5
['C', 'PALMA', 'Mz.', '34', 'Lt.', '5']
10137: Mz 112 Lte 56 C. CALLE 3A. CERRADA DE BOSQUES NORDICOS VIV 1 3
['Mz.', '112', 'Lt.', '56', 'Calle', 'Calle', '3', 'A.', 'CERRADA', 'DE', 'BOSQUES', 'NORDICOS', 'Vivienda', '1', '3']
14155: MARTIRES DE CANANEA 3313
['MARTIRES', 'DE', 'CANANEA', '3313']
33106: CIRCUITO PERSILES 136-A LT 63 MZ 1
['CIRCUITO', 'PERSILES', '136', 'A', 'Lt.', '63', 'Mz.', '1']
13656: PRIV DE ORIENTE 12 35 DPTO 2 COL CENTRO
['PRIV', 'DE', 'ORIENTE', '12', '35', 'DPTO', '2', 'CENTRO']
7624: Francisco Cesar Morales , Dpto. 503, Ed. Fuentes de Minerva. No. 61
['Francisco', 'Cesar', 'Morales', 'Dpto.', '503', 'Ed.', 'Fuentes', 'de', 'Minerva.', 'No.', '61']
39593: 4,2,ISLAS BAHAMAS,MISION VILLAMAR II,QUI
['4', '2', 'ISLAS', 'BA

## TEST QUITAR REDUNDANCIAS

In [20]:
columnas = ["ESTADO", "MUNICIPIO", "COLONIA", "CP"]

for i in range(500):
    np.random.seed(i+1105*i)
    test_adress = np.random.randint(0, direcciones.shape[0]-1)
    red_ptrns = redundant_patterns(direcciones, columnas, test_adress)
    print("~~ DIRECCIÓN ~~")
    print(f"{test_adress}: {direcciones["DIRECCIÓN GARANTÍA"][test_adress]}")
    #print("~~ REDUNDANCIAS ~~")
    #print([patron.patron for patron in red_ptrns])
    print("~~ CLEAN ~~")
    clean_tokens = clean(raw_adress = direcciones["DIRECCIÓN GARANTÍA"][test_adress], patterns= PATRONES + red_ptrns)
    print(" ".join(clean_tokens))
    #time.sleep(10)

~~ DIRECCIÓN ~~
2732: MANZANA 24 LOTE 24 PRIMERA ETAPA AV SAETIN FRACC TORREMOLINOS DOCTOR
~~ CLEAN ~~
Mz. 24 Lt. 24 PRIMERA ETAPA Av. SAETIN TORREMOLINOS DOCTOR
~~ DIRECCIÓN ~~
36157: NICARAGUA, NUM EXT 433, M-13, L-1
~~ CLEAN ~~
NICARAGUA No. Ext. 433 Mz. 13 Lt. 1
~~ DIRECCIÓN ~~
25125: CALLE RIGEL LT 3 MZA 73 FRACC ESTRELLA DE BUENAVISTA CP 86127
~~ CLEAN ~~
Calle RIGEL Lt. 3 Mz. 73 ESTRELLA DE BUENAVISTA
~~ DIRECCIÓN ~~
12874: C INDEPENDENCIA 51 LOC RANCHO NUEVO DE
~~ CLEAN ~~
C INDEPENDENCIA 51 LOC RANCHO NUEVO DE
~~ DIRECCIÓN ~~
22495: HACIENDA SAN RAFAEL No. 214, UP 42, MANZANA 125, FRACCION 12, COL.  HACIENDA REA
~~ CLEAN ~~
HACIENDA SAN RAFAEL No. 214 UP 42 Mz. 125 FRACCION 12 HACIENDA REA
~~ DIRECCIÓN ~~
28066: SEXTO CERRADA DE BOSQUES DE MEXICO, NUM EXT 4, M-106, L-22
~~ CLEAN ~~
SEXTO CERRADA DE BOSQUES DE MEXICO No. Ext. 4 Mz. 106 Lt. 22
~~ DIRECCIÓN ~~
18825: C AXAYACATL 105
~~ CLEAN ~~
C AXAYACATL 105
~~ DIRECCIÓN ~~
2463: SAN PEDRO 61
~~ CLEAN ~~
SAN PEDRO 61
~~ DIRECCI

In [17]:
" ".join(clean_tokens)

'Lt. 66 DE LA Mz. 6 DE LA Calle ORQUIDEAS'